In [19]:
# Training

In [20]:
import pandas as pd
import numpy as np
import sagemaker
from sagemaker import Session
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator

session = sagemaker.Session()
role = sagemaker.get_execution_role()

bucket = "prognostica-cancer-project"
prefix = "features"


In [26]:
!pip install imbalanced-learn


  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)
Using cached sklearn_compat-0.1.5-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn][imbalanced-learn]


In [27]:
dtrain_path_original = f"s3://{bucket}/{prefix}/lung_train.csv"
train_df = pd.read_csv(train_path_original)

train_df.head()


,gender,age,smoking,yellow_fingers,anxiety,peer_pressure,chronic_disease,fatigue,allergy,wheezing,alcohol_consuming,coughing,shortness_of_breath,swallowing_difficulty,chest_pain,lung_cancer,age_group_41-50,age_group_51-60,age_group_61-70,age_group_71+
0,M,60,1,2,1,2,1,2,2,1,1,2,1,1,2,YES,False,True,False,False
1,F,55,2,1,2,2,2,1,2,2,1,1,2,1,2,YES,False,True,False,False
2,M,53,2,2,1,2,2,1,1,1,1,2,2,1,2,YES,False,True,False,False
3,F,70,1,2,1,2,1,2,1,2,2,2,1,1,2,YES,False,False,True,False
4,F,62,1,2,2,2,1,2,2,1,2,1,2,1,2,YES,False,False,True,False


In [28]:
# Load orginal training data

In [29]:
train_path_original = f"s3://{bucket}/{prefix}/lung_train.csv"
train_df = pd.read_csv(train_path_original)

train_df.head()


,gender,age,smoking,yellow_fingers,anxiety,peer_pressure,chronic_disease,fatigue,allergy,wheezing,alcohol_consuming,coughing,shortness_of_breath,swallowing_difficulty,chest_pain,lung_cancer,age_group_41-50,age_group_51-60,age_group_61-70,age_group_71+
0,M,60,1,2,1,2,1,2,2,1,1,2,1,1,2,YES,False,True,False,False
1,F,55,2,1,2,2,2,1,2,2,1,1,2,1,2,YES,False,True,False,False
2,M,53,2,2,1,2,2,1,1,1,1,2,2,1,2,YES,False,True,False,False
3,F,70,1,2,1,2,1,2,1,2,2,2,1,1,2,YES,False,False,True,False
4,F,62,1,2,2,2,1,2,2,1,2,1,2,1,2,YES,False,False,True,False


In [30]:
# Encode and  build balanced dataset

In [39]:
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode you
X_encoded = pd.get_dummies(X)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# SAVE CSV WITHOUT HEADER SINCE CAUSES ERROR 

balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

# Move label to first column
cols = ["lung_cancer"] + [c for c in balanced_df.columns if c != "lung_cancer"]
balanced_df = balanced_df[cols]


session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path



's3://prognostica-cancer-project/features/lung_train_balanced.csv'

In [40]:
print(balanced_df["lung_cancer"].unique())


[1 0]


In [41]:
check_df = pd.read_csv(train_path, header=None)
check_df.iloc[:, -1].value_counts()


20
1    11908
0    11908
Name: count, dtype: int64

In [42]:
# Wrap in Traininginput

In [43]:
train_input = TrainingInput(train_path, content_type="text/csv")


In [44]:
# Define XGBoost estimator

In [45]:
container = sagemaker.image_uris.retrieve("xgboost", session.boto_region_name, "1.5-1")

xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}/output"
)

xgb.set_hyperparameters(
    objective="binary:logistic",
    num_round=500,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    eval_metric="auc",
    scale_pos_weight=1
)


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [46]:
# Train the model
xgb.fit({"train": train_input})



INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-03-22-17-38-22-873


2026-03-22 17:38:24 Starting - Starting the training job...
2026-03-22 17:38:39 Starting - Preparing the instances for training...
2026-03-22 17:39:02 Downloading - Downloading input data...
2026-03-22 17:39:47 Downloading - Downloading the training image......
2026-03-22 17:40:59 Training - Training image download completed. Training in progress.
2026-03-22 17:40:59 Uploading - Uploading generated training model./miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-03-22 17:40:52.339 ip-10-0-235-69.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-22 17:40:52.361 ip-10-0-235-69.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-22:17:40:52:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-22:1

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # Train the model                                                                            │
│ ❱ 2 xgb.fit({"train": train_input})                                                              │
│   3                                                                                              │
│   4                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:167 in wrapper  │
│                                                                                                  │
│   164 │   │   │   │   │   caught_ex = e                                                          │
│   165 │   │   │   │   finally:                                                                   │
│   166 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 167 │   │   │   │   │   │   raise caught_ex                                                    │
│   168 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   169 │   │   │   else:                                                                          │
│   170 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:138 in wrapper  │
│                                                                                                  │
│   135 │   │   │   │   start_timer = perf_counter()                                               │
│   136 │   │   │   │   try:                                                                       │
│   137 │   │   │   │   │   # Call the original function                                           │
│ ❱ 138 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   139 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   140 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   141 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:1380 in fit                       │
│                                                            

In [ ]:
# Train the model

In [ ]:
xgb.fit({"train": train_input})


In [ ]:
# Download model.tar.gz

In [ ]:
import tarfile
import os

model_artifact = xgb.model_data
model_artifact


In [ ]:
#Download
!aws s3 cp {model_artifact} model.tar.gz


In [ ]:
# Extract
tar = tarfile.open("model.tar.gz")
tar.extractall()
tar.close()


In [ ]:
# Load the trained model

In [ ]:
import xgboost as xgb

model = xgb.Booster()
model.load_model("xgboost-model")


In [ ]:
# Load test data

In [ ]:
test_path = f"s3://{bucket}/{prefix}/lung_test.csv"
test_df = pd.read_csv(test_path)

X_test = test_df.drop("lung_cancer", axis=1)
y_test = test_df["lung_cancer"].map({"YES": 1, "NO": 0})

X_test_encoded = pd.get_dummies(X_test)

# Align columns with training
X_test_encoded = X_test_encoded.reindex(columns=X_balanced.columns, fill_value=0)


In [ ]:
# Predict

In [ ]:
dtest = xgb.DMatrix(X_test_encoded)
y_pred_prob = model.predict(dtest)
y_pred = (y_pred_prob > 0.5).astype(int)


In [ ]:
# Evaluate

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode
X_encoded = pd.get_dummies(X)

# Convert any boolean columns to integers
X_encoded = X_encoded.astype(int)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# Save CSV WITHOUT header
balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode
X_encoded = pd.get_dummies(X)

# Convert any boolean columns to integers
X_encoded = X_encoded.astype(int)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# Save CSV WITHOUT header
balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode
X_encoded = pd.get_dummies(X)

# Convert any boolean columns to integers
X_encoded = X_encoded.astype(int)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# Save CSV WITHOUT header
balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode
X_encoded = pd.get_dummies(X)

# Convert any boolean columns to integers
X_encoded = X_encoded.astype(int)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# Save CSV WITHOUT header
balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode
X_encoded = pd.get_dummies(X)

# Convert any boolean columns to integers
X_encoded = X_encoded.astype(int)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# Save CSV WITHOUT header
balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode
X_encoded = pd.get_dummies(X)

# Convert any boolean columns to integers
X_encoded = X_encoded.astype(int)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# Save CSV WITHOUT header
balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode
X_encoded = pd.get_dummies(X)

# Convert any boolean columns to integers
X_encoded = X_encoded.astype(int)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# Save CSV WITHOUT header
balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path
from imblearn.over_sampling import SMOTE

# Separate features and target
X = train_df.drop("lung_cancer", axis=1)
y = train_df["lung_cancer"].map({"YES": 1, "NO": 0})

# One-hot encode
X_encoded = pd.get_dummies(X)

# Convert any boolean columns to integers
X_encoded = X_encoded.astype(int)

# Apply SMOTE
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X_encoded, y)

# Build final balanced dataframe
balanced_df = pd.concat([X_balanced, y_balanced.rename("lung_cancer")], axis=1)
balanced_df["lung_cancer"] = balanced_df["lung_cancer"].astype(int)

# Save CSV WITHOUT header
balanced_df.to_csv("lung_train_balanced.csv", index=False, header=False)

session.upload_data(
    "lung_train_balanced.csv",
    bucket=bucket,
    key_prefix=prefix
)

train_path = f"s3://{bucket}/{prefix}/lung_train_balanced.csv"
train_path
